In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate bitsandbytes peft datasets pillow pandas tqdm

import os
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("/content/drive/MyDrive/DL_Final_DATA")

STUDENT_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
TEACHER_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

MAX_CHOICES = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train (1).csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test (1).csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print("✅ CSVs loaded | train:", train_df.shape, "val:", val_df.shape, "test:", test_df.shape)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.6 MB/s eta 0:00:00
✅ CSVs loaded | train: (3109, 15) val: (1048, 15) test: (1008, 13)


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

print("Loading Qwen teacher model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

teacher_processor = AutoProcessor.from_pretrained(TEACHER_MODEL_ID)
teacher_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    TEACHER_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
teacher_model.eval()
print("✅ Teacher loaded")

Loading Qwen teacher model...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Teacher loaded


In [ ]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_caption_prompt(row):
    subject = safe_text(row.get("subject", ""))
    topic   = safe_text(row.get("topic", ""))

    subject_hint = ""
    if subject:
        subject_hint = f" This image is from a {subject}"
        if topic:
            subject_hint += f" question about {topic}"
        subject_hint += "."

    prompt = f"""Describe this image precisely in 4-6 sentences.{subject_hint}

CRITICAL: Extract ALL visible text exactly as it appears, including:
- Every label, title, caption, and annotation
- Every axis title, axis value, legend entry, and data label on charts/graphs
- Every label and arrow caption on diagrams
- Every place name, region label, and date on maps and timelines
- Every name and number in tables

Then briefly describe the visual structure: chart type, diagram layout, color coding, what is connected to what, key shapes or organisms.

Be exact and exhaustive about text. Do not paraphrase text values. Do not interpret what the image means or answer any question — only describe what is visible."""
    return prompt.strip()


print("✅ Caption prompt builder ready")
print("\n--- Sample caption prompt (train row 0) ---")
print(build_caption_prompt(train_df.iloc[0]))

✅ Caption prompt builder ready

--- Sample caption prompt (train row 0) ---
Describe this image precisely in 4-6 sentences. This image is from a natural science question about literacy-in-science.

CRITICAL: Extract ALL visible text exactly as it appears, including:
- Every label, title, caption, and annotation
- Every axis title, axis value, legend entry, and data label on charts/graphs
- Every label and arrow caption on diagrams
- Every place name, region label, and date on maps and timelines
- Every name and number in tables

Then briefly describe the visual structure: chart type, diagram layout, color coding, what is connected to what, key shapes or organisms.

Be exact and exhaustive about text. Do not paraphrase text values. Do not interpret what the image means or answer any question — only describe what is visible.


In [ ]:



@torch.no_grad()
def generate_caption(row, max_new_tokens=200):
    img_path = DATA_DIR / row["image_path"]
    img = Image.open(img_path).convert("RGB")
    prompt = build_caption_prompt(row)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt},
        ],
    }]

    text = teacher_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = teacher_processor(
        text=[text], images=[img], return_tensors="pt"
    ).to(teacher_model.device)

    out_ids = teacher_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        temperature=1.0,
        pad_token_id=teacher_processor.tokenizer.eos_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    generated = out_ids[0, input_len:]
    caption = teacher_processor.decode(generated, skip_special_tokens=True).strip()
    return caption


print("✅ Updated caption prompt")

✅ Updated caption prompt


In [ ]:
# Sample diverse rows: different subjects, different topics
sample_indices = []
seen_subjects = set()
for idx, row in train_df.iterrows():
    subj_topic = (row.get("subject", ""), row.get("topic", ""))
    if subj_topic not in seen_subjects and len(sample_indices) < 8:
        seen_subjects.add(subj_topic)
        sample_indices.append(idx)

print(f"Sampling {len(sample_indices)} diverse rows by subject/topic\n")

for idx in sample_indices:
    row = train_df.iloc[idx]
    caption = generate_caption(row)
    print(f"=== train_df[{idx}] ===")
    print(f"ID:       {row['id']}")
    print(f"Subject:  {row.get('subject', '')} | Topic: {row.get('topic', '')}")
    print(f"Question: {row['question'][:120]}")
    print(f"Caption:  {caption}")
    print()

Sampling 8 diverse rows by subject/topic

=== train_df[0] ===
ID:       train_07667
Subject:  natural science | Topic: literacy-in-science
Question: Why might putting each tadpole in its own pool of water increase the reproductive success of a male Amazonian poison fro
Caption:  The image shows two poison dart frogs with distinct patterns. The frog in the foreground has a black body with yellow stripes and spots, while the frog in the background has a green body with black spots. Both frogs have long legs and are positioned on a green surface. There is no visible text or labels in the image.

=== train_df[7] ===
ID:       train_03216
Subject:  language science | Topic: reading-comprehension
Question: Based on the text, which of the following things made the passenger pigeon migration a special event?
Caption:  The image shows a bird perched on a branch. The bird has a grayish-brown plumage with a lighter underside. Its beak is short and pointed, and its tail is long and slightly forked

In [ ]:
caption_path = DATA_DIR / "image_captions_v5.csv"

# Resume support: if we already have a caption file, load and skip done IDs
existing_captions = {}
if caption_path.exists():
    existing_df = pd.read_csv(caption_path)
    existing_captions = dict(zip(existing_df["id"], existing_df["caption"]))
    print(f"✅ Resuming — {len(existing_captions)} captions already done")
else:
    print("Starting fresh caption generation")


def caption_split(df, split_name):
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Captioning {split_name}"):
        rid = row["id"]
        if rid in existing_captions:
            rows.append({"id": rid, "split": split_name, "caption": existing_captions[rid]})
            continue

        try:
            cap = generate_caption(row)
        except Exception as e:
            print(f"  ⚠ Error on {rid}: {e}")
            cap = ""   # fall back to empty caption rather than crashing

        rows.append({"id": rid, "split": split_name, "caption": cap})

        # Checkpoint every 200 captions
        if (idx + 1) % 200 == 0:
            partial = pd.DataFrame(rows)
            partial.to_csv(caption_path.with_suffix(".partial.csv"), index=False)

    return rows


all_rows = []
all_rows.extend(caption_split(train_df, "train"))

# Save train captions before continuing
pd.DataFrame(all_rows).to_csv(caption_path, index=False)
print(f"✅ Train captions saved | total so far: {len(all_rows)}")

all_rows.extend(caption_split(val_df, "val"))
pd.DataFrame(all_rows).to_csv(caption_path, index=False)
print(f"✅ Val captions saved | total so far: {len(all_rows)}")

all_rows.extend(caption_split(test_df, "test"))
captions_df = pd.DataFrame(all_rows)
captions_df.to_csv(caption_path, index=False)

print(f"\n✅ All captions saved: {caption_path}")
print(f"Total captions: {len(captions_df)}")
print("\nSplit breakdown:")
print(captions_df["split"].value_counts())
print(f"\nMean caption length: {captions_df['caption'].str.len().mean():.0f} chars")
print(f"Empty captions:       {(captions_df['caption'] == '').sum()}")

Starting fresh caption generation


Captioning train:   0%|          | 0/3109 [00:00<?, ?it/s]

✅ Train captions saved | total so far: 3109


Captioning val:   0%|          | 0/1048 [00:00<?, ?it/s]

✅ Val captions saved | total so far: 4157


Captioning test:   0%|          | 0/1008 [00:00<?, ?it/s]


✅ All captions saved: /content/drive/MyDrive/DL_Final_DATA/image_captions_v5.csv
Total captions: 5165

Split breakdown:
split
train    3109
val      1048
test     1008
Name: count, dtype: int64

Mean caption length: 423 chars
Empty captions:       0


In [ ]:
caption_path = DATA_DIR / "image_captions_v5.csv"
captions_df = pd.read_csv(caption_path)

# Build a single id -> caption lookup. The split column doesn't matter for the merge —
# every image_id appears once.
caption_lookup = dict(zip(captions_df["id"], captions_df["caption"].fillna("")))

def attach_caption(df):
    df = df.copy()
    df["caption"] = df["id"].map(caption_lookup).fillna("")
    return df

train_df = attach_caption(train_df)
val_df   = attach_caption(val_df)
test_df  = attach_caption(test_df)

# Sanity
for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = (df["caption"] == "").sum()
    print(f"{split_name}: {len(df)} rows, {missing} with empty caption")

print("\n✅ Captions attached. Sample:")
print(f"  ID:       {train_df.iloc[0]['id']}")
print(f"  Question: {train_df.iloc[0]['question'][:80]}")
print(f"  Caption:  {train_df.iloc[0]['caption'][:200]}...")

train: 3109 rows, 0 with empty caption
val: 1048 rows, 0 with empty caption
test: 1008 rows, 0 with empty caption

✅ Captions attached. Sample:
  ID:       train_07667
  Question: Why might putting each tadpole in its own pool of water increase the reproductiv
  Caption:  The image shows two poison dart frogs with distinct patterns. The frog in the foreground has a black body with yellow stripes and spots, while the frog in the background has a green body with black sp...


In [ ]:
import numpy as np

teacher_train_df = pd.read_csv(DATA_DIR / "teacher_train_probs_logits_v2_new.csv")
teacher_val_df   = pd.read_csv(DATA_DIR / "teacher_val_probs_logits_v2.csv")

teacher_carry_cols = (
    ["id", "teacher_answer", "teacher_confidence"]
    + [f"p{i}"     for i in range(MAX_CHOICES)]
    + [f"logit{i}" for i in range(MAX_CHOICES)]
)

train_kd_df = train_df.merge(teacher_train_df[teacher_carry_cols], on="id", how="left")
val_kd_df   = val_df.merge(teacher_val_df[teacher_carry_cols],     on="id", how="left")

CONF_THRESHOLD = 0.7

def build_soft_target(row):
    nc = int(row["num_choices"])
    gt = int(row["answer"])
    teacher_pred = int(row["teacher_answer"])
    conf = float(row["teacher_confidence"])

    target = np.full(MAX_CHOICES, np.nan, dtype=np.float32)
    teacher_probs = np.array([row[f"p{i}"] for i in range(nc)], dtype=np.float32)

    use_one_hot = (teacher_pred != gt) or (conf < CONF_THRESHOLD)
    if use_one_hot:
        target[:nc] = 0.0
        target[gt]  = 1.0
    else:
        target[:nc] = teacher_probs
    return target

soft_targets = np.stack([build_soft_target(r) for _, r in train_kd_df.iterrows()])
for i in range(MAX_CHOICES):
    train_kd_df[f"soft{i}"] = soft_targets[:, i]

n_total          = len(train_kd_df)
n_teacher_wrong  = (train_kd_df["teacher_answer"] != train_kd_df["answer"]).sum()
n_teacher_unsure = (
    (train_kd_df["teacher_answer"] == train_kd_df["answer"])
    & (train_kd_df["teacher_confidence"] < CONF_THRESHOLD)
).sum()
n_kept_soft = n_total - n_teacher_wrong - n_teacher_unsure
print(f"Train: {n_total} | one-hot: {n_teacher_wrong+n_teacher_unsure} | kept teacher: {n_kept_soft}")

Train: 3109 | one-hot: 199 | kept teacher: 2910


In [ ]:
def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


# Rough character budget for non-essential context (lecture + caption combined).
# Question, choices, metadata, hint, and the "Answer:" suffix are essential.
# This is a soft cap; if exceeded, we truncate the lecture first.
MAX_LECTURE_CHARS = 800     # cap lecture text individually
MAX_CAPTION_CHARS = 600     # cap caption individually


def truncate_at_sentence(text, max_chars):
    if len(text) <= max_chars:
        return text
    # Try to cut at the last sentence boundary before max_chars
    cut = text[:max_chars]
    last_period = max(cut.rfind(". "), cut.rfind(".\n"), cut.rfind("! "), cut.rfind("? "))
    if last_period > max_chars * 0.5:   # only use sentence break if it's not too short
        return cut[:last_period + 1]
    return cut + "..."


def build_student_messages(row):
    """
    v5 student prompt: same as v4 plus a Caption section between Metadata
    and Context. Caption gives the student access to text/structure the
    teacher (Qwen2.5-VL-7B) extracted from the image.
    """
    hint    = safe_text(row.get("hint", ""))
    lecture = safe_text(row.get("lecture", ""))
    caption = safe_text(row.get("caption", ""))

    # Truncate long fields to keep prompt manageable
    lecture = truncate_at_sentence(lecture, MAX_LECTURE_CHARS) if lecture else ""
    caption = truncate_at_sentence(caption, MAX_CAPTION_CHARS) if caption else ""

    meta_parts = []
    for col in ["grade", "subject", "topic", "category", "skill"]:
        val = safe_text(row.get(col, ""))
        if val:
            meta_parts.append(f"{col}: {val}")
    meta_text = "\n".join(meta_parts)

    context_parts = []
    if lecture: context_parts.append("Lecture:\n" + lecture)
    if hint:    context_parts.append("Hint:\n" + hint)
    context_text = "\n\n".join(context_parts)

    choices = row["choices"]
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    # Build the text block. Caption is a NEW section in v5.
    sections = []
    sections.append(f"You are solving a science multiple-choice question.")
    if meta_text:
        sections.append(f"Metadata:\n{meta_text}")
    if caption:
        sections.append(f"Image description:\n{caption}")
    if context_text:
        sections.append(context_text)
    sections.append(f"Question:\n{row['question']}")
    sections.append(f"Choices:\n{choices_text}")
    sections.append("Answer:")

    text_block = "\n\n".join(sections)

    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": text_block},
        ],
    }]


print("✅ v5 student message builder ready (with caption + truncation)")
print("\n--- Sample v5 prompt ---")
sample_messages = build_student_messages(train_kd_df.iloc[0])
print(sample_messages[0]["content"][1]["text"][:1500])

✅ v5 student message builder ready (with caption + truncation)

--- Sample v5 prompt ---
You are solving a science multiple-choice question.

Metadata:
grade: grade8
subject: natural science
topic: literacy-in-science
category: Adaptations and natural selection
skill: How can animal behaviors affect reproductive success? Identify evidence to support a claim

Image description:
The image shows two poison dart frogs with distinct patterns. The frog in the foreground has a black body with yellow stripes and spots, while the frog in the background has a green body with black spots. Both frogs have long legs and are positioned on a green surface. There is no visible text or labels in the image.

Lecture:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make s

In [ ]:
import gc
try:
    del student_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Loading SmolVLM student (fresh) for v5...")

student_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

student_processor = AutoProcessor.from_pretrained(STUDENT_MODEL_ID)
student_processor.tokenizer.padding_side = "left"
if student_processor.tokenizer.pad_token is None:
    student_processor.tokenizer.pad_token = student_processor.tokenizer.eos_token

student_model = AutoModelForImageTextToText.from_pretrained(
    STUDENT_MODEL_ID,
    quantization_config=student_bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
student_model.config.use_cache = False
student_model = prepare_model_for_kbit_training(student_model)

# Same LoRA config as v4 (the one that worked: r=8, alpha=16, attn + MLP)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} | Headroom: {5_000_000 - trainable:,}")
assert trainable <= 5_000_000
print("✅ v5 student ready")

Loading SmolVLM student (fresh) for v5...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339
Trainable: 4,784,128 | Headroom: 215,872
✅ v5 student ready


In [ ]:
def collect_digit_token_ids(tokenizer, digit):
    candidates = set()
    for s in [str(digit), f" {digit}", f"\n{digit}", f"\n\n{digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            candidates.add(ids[0])
    for s in [str(digit), f" {digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) >= 1:
            candidates.add(ids[-1])
    return sorted(candidates)


CHOICE_TOKEN_VARIANTS = {}
for d in range(MAX_CHOICES):
    ids = collect_digit_token_ids(student_processor.tokenizer, d)
    CHOICE_TOKEN_VARIANTS[d] = ids

MAX_VARIANTS = max(len(v) for v in CHOICE_TOKEN_VARIANTS.values())
CHOICE_TOKEN_MATRIX = torch.full((MAX_CHOICES, MAX_VARIANTS), -1, dtype=torch.long)
for d, ids in CHOICE_TOKEN_VARIANTS.items():
    for j, tid in enumerate(ids):
        CHOICE_TOKEN_MATRIX[d, j] = tid

CHOICE_TOKEN_MATRIX_DEVICE = None
print("CHOICE_TOKEN_MATRIX:")
print(CHOICE_TOKEN_MATRIX)

CHOICE_TOKEN_MATRIX:
tensor([[32],
        [33],
        [34],
        [35],
        [36]])


In [ ]:
class KDScienceDataset(torch.utils.data.Dataset):
    def __init__(self, df, data_dir, use_soft_targets=True):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.use_soft_targets = use_soft_targets

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image    = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        messages = build_student_messages(row)

        num_choices = int(row["num_choices"])
        answer = int(row["answer"]) if "answer" in row and not pd.isna(row.get("answer", np.nan)) else -1

        teacher_probs = np.zeros(MAX_CHOICES, dtype=np.float32)
        if self.use_soft_targets:
            for i in range(MAX_CHOICES):
                col = f"soft{i}"
                if col in row and not pd.isna(row[col]):
                    teacher_probs[i] = float(row[col])
        if teacher_probs.sum() > 0:
            teacher_probs = teacher_probs / teacher_probs.sum()

        return {
            "id":            row["id"],
            "image":         image,
            "messages":      messages,
            "answer":        answer,
            "num_choices":   num_choices,
            "teacher_probs": teacher_probs,
        }


def kd_collate_fn(batch):
    images = [item["image"] for item in batch]
    texts  = [
        student_processor.apply_chat_template(item["messages"], add_generation_prompt=True)
        for item in batch
    ]
    inputs = student_processor(
        text=texts, images=images, return_tensors="pt", padding=True,
    )
    answers       = torch.tensor([item["answer"]      for item in batch], dtype=torch.long)
    num_choices   = torch.tensor([item["num_choices"] for item in batch], dtype=torch.long)
    teacher_probs = torch.tensor(
        np.stack([item["teacher_probs"] for item in batch]), dtype=torch.float32,
    )
    return {
        "ids":           [item["id"] for item in batch],
        "inputs":        inputs,
        "answers":       answers,
        "num_choices":   num_choices,
        "teacher_probs": teacher_probs,
    }


train_dataset = KDScienceDataset(train_kd_df, DATA_DIR, use_soft_targets=True)
val_dataset   = KDScienceDataset(val_kd_df,   DATA_DIR, use_soft_targets=False)
print("✅ Datasets ready | train:", len(train_dataset), "val:", len(val_dataset))

✅ Datasets ready | train: 3109 val: 1048


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=kd_collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=kd_collate_fn, num_workers=0)

# Sanity: confirm pixel_values present (caught a bug in v4 with this)
batch_check = next(iter(train_loader))
assert "pixel_values" in batch_check["inputs"], "❌ pixel_values missing!"
print(f"✅ Loaders ready | effective batch = {BATCH_SIZE * GRAD_ACCUM_STEPS}")

# ALSO sanity-check that prompts are not absurdly long
sample_text_len = batch_check["inputs"]["input_ids"].shape[1]
print(f"Sample batch token length: {sample_text_len}")
if sample_text_len > 4000:
    print(f"⚠ Prompts are very long ({sample_text_len} tokens). Consider tightening MAX_LECTURE_CHARS / MAX_CAPTION_CHARS.")

✅ Loaders ready | effective batch = 16
Sample batch token length: 1256


In [ ]:
import torch.nn.functional as F

KD_ALPHA       = 0.7
KD_TEMPERATURE = 2.0

def get_choice_logits_from_student(batch):
    global CHOICE_TOKEN_MATRIX_DEVICE
    inputs = {k: (v.to(student_model.device) if torch.is_tensor(v) else v) for k, v in batch["inputs"].items()}
    outputs = student_model(**inputs)
    last_logits = outputs.logits[:, -1, :].float()
    if CHOICE_TOKEN_MATRIX_DEVICE is None or CHOICE_TOKEN_MATRIX_DEVICE.device != last_logits.device:
        CHOICE_TOKEN_MATRIX_DEVICE = CHOICE_TOKEN_MATRIX.to(last_logits.device)
    B = last_logits.size(0)
    C, V = CHOICE_TOKEN_MATRIX_DEVICE.shape
    flat_idx = CHOICE_TOKEN_MATRIX_DEVICE.clamp(min=0).reshape(-1)
    gathered = last_logits[:, flat_idx].reshape(B, C, V)
    valid_mask = (CHOICE_TOKEN_MATRIX_DEVICE >= 0).unsqueeze(0).expand(B, -1, -1)
    gathered = gathered.masked_fill(~valid_mask, float("-inf"))
    return torch.logsumexp(gathered, dim=-1)


def compute_kd_loss(choice_logits, answers, teacher_probs, num_choices):
    device = choice_logits.device
    answers, teacher_probs, num_choices = answers.to(device), teacher_probs.to(device), num_choices.to(device)
    B, C = choice_logits.shape
    arange = torch.arange(C, device=device).unsqueeze(0).expand(B, -1)
    mask = arange < num_choices.unsqueeze(1)
    masked_logits = choice_logits.masked_fill(~mask, -1e9)
    ce_loss = F.cross_entropy(masked_logits, answers)
    student_log_probs = F.log_softmax(masked_logits / KD_TEMPERATURE, dim=-1)
    teacher_probs = teacher_probs.masked_fill(~mask, 0.0)
    teacher_probs = teacher_probs / teacher_probs.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    kd_loss = F.kl_div(student_log_probs, teacher_probs, reduction="batchmean") * (KD_TEMPERATURE ** 2)
    total = KD_ALPHA * ce_loss + (1.0 - KD_ALPHA) * kd_loss
    return total, ce_loss.detach(), kd_loss.detach()


@torch.no_grad()
def evaluate_student():
    student_model.eval()
    all_preds, all_answers, all_ids = [], [], []
    correct = total = 0
    for batch in tqdm(val_loader, desc="Validating"):
        choice_logits = get_choice_logits_from_student(batch)
        num_choices = batch["num_choices"].to(choice_logits.device)
        answers     = batch["answers"].to(choice_logits.device)
        B, C = choice_logits.shape
        arange = torch.arange(C, device=choice_logits.device).unsqueeze(0).expand(B, -1)
        mask = arange < num_choices.unsqueeze(1)
        masked_logits = choice_logits.masked_fill(~mask, -1e9)
        preds = masked_logits.argmax(dim=-1)
        correct += (preds == answers).sum().item()
        total   += answers.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_answers.extend(answers.cpu().tolist())
        all_ids.extend(batch["ids"])
    acc = correct / total
    print(f"Validation Accuracy: {acc*100:.2f}%")
    return acc, pd.DataFrame({"id": all_ids, "answer": all_answers, "pred": all_preds})


print(f"✅ Loss + eval ready | KD_ALPHA={KD_ALPHA} | T={KD_TEMPERATURE}")

✅ Loss + eval ready | KD_ALPHA=0.7 | T=2.0


In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS       = 7
LR           = 2e-4
WEIGHT_DECAY = 0.01

optimizer = AdamW(
    [p for p in student_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)

steps_per_epoch  = len(train_loader) // GRAD_ACCUM_STEPS
total_opt_steps  = steps_per_epoch * EPOCHS
warmup_opt_steps = max(1, int(0.05 * total_opt_steps))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_opt_steps,
    num_training_steps=total_opt_steps,
)
print(f"✅ Optimizer + cosine | epochs={EPOCHS} | total_steps={total_opt_steps} | warmup={warmup_opt_steps}")

✅ Optimizer + cosine | epochs=7 | total_steps=1358 | warmup=67


In [ ]:
print("Evaluating v5 base before training...")
base_acc, _ = evaluate_student()
print(f"v5 base val accuracy: {base_acc*100:.2f}%")

Evaluating v5 base before training...


Validating:   0%|          | 0/524 [00:00<?, ?it/s]

Validation Accuracy: 42.37%
v5 base val accuracy: 42.37%


In [ ]:
best_val_acc = 0.0
best_dir = DATA_DIR / "smolvlm_kd_lora_best_v5"
best_dir.mkdir(exist_ok=True)
global_step = 0

print("Starting v5 KD training (with captions)...")
for epoch in range(EPOCHS):
    print(f"\n================ EPOCH {epoch + 1}/{EPOCHS} ================")
    student_model.train()
    running_loss = running_ce = running_kd = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Train epoch {epoch+1}")):
        choice_logits = get_choice_logits_from_student(batch)
        loss, ce_loss, kd_loss = compute_kd_loss(
            choice_logits=choice_logits,
            answers=batch["answers"],
            teacher_probs=batch["teacher_probs"],
            num_choices=batch["num_choices"],
        )
        (loss / GRAD_ACCUM_STEPS).backward()
        running_loss += loss.item()
        running_ce   += ce_loss.item()
        running_kd   += kd_loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        if (step + 1) % 100 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"Step {step+1} | Loss: {running_loss/100:.4f} | CE: {running_ce/100:.4f} | "
                  f"KD: {running_kd/100:.4f} | LR: {current_lr:.2e}")
            running_loss = running_ce = running_kd = 0.0

    print(f"\nEvaluating after epoch {epoch + 1}")
    val_acc, val_pred_df = evaluate_student()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print("✅ New best — saving to:", best_dir)
        student_model.save_pretrained(best_dir)
        student_processor.save_pretrained(best_dir)
        val_pred_df.to_csv(DATA_DIR / "val_predictions_best_v5.csv", index=False)

    print(f"Best val acc so far: {best_val_acc*100:.2f}%")

print(f"\n✅ v5 training complete | Best Val: {best_val_acc*100:.2f}%")

Starting v5 KD training (with captions)...

================ EPOCH 1/7 ================


Train epoch 1:   0%|          | 0/1555 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step 100 | Loss: 1.9326 | CE: 1.1844 | KD: 3.6782 | LR: 3.58e-05
Step 200 | Loss: 1.7809 | CE: 1.0380 | KD: 3.5143 | LR: 7.46e-05
Step 300 | Loss: 1.6804 | CE: 0.9822 | KD: 3.3097 | LR: 1.10e-04
Step 400 | Loss: 1.5679 | CE: 0.9202 | KD: 3.0790 | LR: 1.49e-04
Step 500 | Loss: 1.4454 | CE: 0.8485 | KD: 2.8382 | LR: 1.85e-04
Step 600 | Loss: 1.3346 | CE: 0.7810 | KD: 2.6264 | LR: 2.00e-04
Step 700 | Loss: 1.2400 | CE: 0.7491 | KD: 2.3856 | LR: 2.00e-04
Step 800 | Loss: 1.2293 | CE: 0.7167 | KD: 2.4253 | LR: 2.00e-04
Step 900 | Loss: 1.0182 | CE: 0.6052 | KD: 1.9819 | LR: 1.99e-04
Step 1000 | Loss: 0.9205 | CE: 0.5527 | KD: 1.7785 | LR: 1.99e-04
Step 1100 | Loss: 1.2854 | CE: 0.7909 | KD: 2.4392 | LR: 1.99e-04
Step 1200 | Loss: 0.9711 | CE: 0.5245 | KD: 2.0132 | LR: 1.98e-04
Step 1300 | Loss: 1.0788 | CE: 0.6422 | KD: 2.0977 | LR: 1.97e-04
Step 1400 | Loss: 1.1036 | CE: 0.6547 | KD: 2.1513 | LR: 1.97e-04
Step 1500 | Loss: 1.0883 | CE: 0.6453 | KD: 2.1220 | LR: 1.96e-04

Evaluating after e

Validating:   0%|          | 0/524 [00:00<?, ?it/s]

...

Validation Accuracy: 80.34%
✅ New best — saving to: /content/drive/MyDrive/DL_Final_DATA/smolvlm_kd_lora_best_v5
Best val acc so far: 80.34%

================ EPOCH 7/7 ================


Train epoch 7:   0%|          | 0/1555 [00:00<?, ?it/s]

Step 100 | Loss: 0.1937 | CE: 0.0894 | KD: 0.4371 | LR: 9.65e-06
Step 200 | Loss: 0.1821 | CE: 0.0773 | KD: 0.4266 | LR: 8.34e-06
Step 300 | Loss: 0.1129 | CE: 0.0441 | KD: 0.2736 | LR: 7.21e-06
Step 400 | Loss: 0.1782 | CE: 0.0870 | KD: 0.3912 | LR: 6.08e-06
Step 500 | Loss: 0.2040 | CE: 0.1017 | KD: 0.4428 | LR: 5.11e-06
Step 600 | Loss: 0.1847 | CE: 0.0916 | KD: 0.4020 | LR: 4.16e-06
Step 700 | Loss: 0.1141 | CE: 0.0355 | KD: 0.2977 | LR: 3.37e-06
Step 800 | Loss: 0.2071 | CE: 0.0904 | KD: 0.4793 | LR: 2.60e-06
Step 900 | Loss: 0.1380 | CE: 0.0561 | KD: 0.3293 | LR: 1.98e-06
Step 1000 | Loss: 0.1225 | CE: 0.0504 | KD: 0.2907 | LR: 1.41e-06
Step 1100 | Loss: 0.1699 | CE: 0.0712 | KD: 0.4002 | LR: 9.60e-07
Step 1200 | Loss: 0.1278 | CE: 0.0466 | KD: 0.3172 | LR: 5.73e-07
Step 1300 | Loss: 0.1733 | CE: 0.0735 | KD: 0.4064 | LR: 3.03e-07
Step 1400 | Loss: 0.2110 | CE: 0.1035 | KD: 0.4619 | LR: 1.07e-07
Step 1500 | Loss: 0.2044 | CE: 0.0932 | KD: 0.4640 | LR: 1.45e-08

Evaluating after e

Validating:   0%|          | 0/524 [00:00<?, ?it/s]

Validation Accuracy: 80.44%
✅ New best — saving to: /content/drive/MyDrive/DL_Final_DATA/smolvlm_kd_lora_best_v5
Best val acc so far: 80.44%

✅ v5 training complete | Best Val: 80.44%


In [ ]:
class TestScienceDataset(torch.utils.data.Dataset):
    def __init__(self, df, data_dir):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image    = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        messages = build_student_messages(row)
        return {
            "id": row["id"],
            "image": image,
            "messages": messages,
            "num_choices": int(row["num_choices"]),
        }


def test_collate_fn(batch):
    images = [item["image"] for item in batch]
    texts  = [
        student_processor.apply_chat_template(item["messages"], add_generation_prompt=True)
        for item in batch
    ]
    inputs = student_processor(
        text=texts, images=images, return_tensors="pt", padding=True,
    )
    return {
        "ids":         [item["id"] for item in batch],
        "inputs":      inputs,
        "num_choices": torch.tensor([item["num_choices"] for item in batch], dtype=torch.long),
    }


test_dataset = TestScienceDataset(test_df, DATA_DIR)
test_loader  = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=test_collate_fn, num_workers=0,
)
print("✅ Test loader ready | samples:", len(test_dataset))

✅ Test loader ready | samples: 1008


In [ ]:
import os
v5_dir = DATA_DIR / "smolvlm_kd_lora_best_v5"
v4_dir = DATA_DIR / "smolvlm_kd_lora_best_v4_fixed"  # or wherever you saved the fixed v4

print("v5 dir exists:", v5_dir.exists(), "| files:", os.listdir(v5_dir) if v5_dir.exists() else "—")
print("v4 dir exists:", v4_dir.exists(), "| files:", os.listdir(v4_dir) if v4_dir.exists() else "—")

v5 dir exists: True | files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']
v4 dir exists: True | files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'processor_config.json']


In [ ]:
import gc
for var in ["student_model", "teacher_model", "teacher_processor", "base_student"]:
    try:
        exec(f"del {var}")
    except NameError:
        pass
gc.collect()
torch.cuda.empty_cache()
print(f"GPU after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU after cleanup: 0.38 GB


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

student_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load base SmolVLM ONCE — both adapters will share it
print("Loading base SmolVLM...")
base_student = AutoModelForImageTextToText.from_pretrained(
    STUDENT_MODEL_ID,
    quantization_config=student_bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
base_student.config.use_cache = False
print("✅ Base SmolVLM loaded")

# Processor for v5 (with caption-aware padding etc — same as v4)
student_processor = AutoProcessor.from_pretrained(STUDENT_MODEL_ID)
student_processor.tokenizer.padding_side = "left"
if student_processor.tokenizer.pad_token is None:
    student_processor.tokenizer.pad_token = student_processor.tokenizer.eos_token
print("✅ Processor configured")

Loading base SmolVLM...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

✅ Base SmolVLM loaded
✅ Processor configured


In [ ]:
# Load v4 first as the "default" adapter
student_model = PeftModel.from_pretrained(
    base_student,
    DATA_DIR / "smolvlm_kd_lora_best_v4_fixed",
    adapter_name="v4",
)

# Add v5 as a second adapter on the same model
student_model.load_adapter(
    DATA_DIR / "smolvlm_kd_lora_best_v5",
    adapter_name="v5",
)

# Verify both adapters loaded with non-zero weights
print("Adapters available:", list(student_model.peft_config.keys()))

# Sanity check both adapters have trained weights
def check_adapter_alive(adapter_name):
    student_model.set_adapter(adapter_name)
    nonzero_lora_b = 0
    total_lora_b = 0
    for name, p in student_model.named_parameters():
        if "lora_B" in name and adapter_name in name:
            total_lora_b += 1
            if p.float().abs().max().item() > 1e-6:
                nonzero_lora_b += 1
    print(f"  Adapter '{adapter_name}': {nonzero_lora_b}/{total_lora_b} lora_B modules have non-zero weights")
    return nonzero_lora_b > 0

assert check_adapter_alive("v4"), "❌ v4 adapter has all-zero LoRA — load failed"
assert check_adapter_alive("v5"), "❌ v5 adapter has all-zero LoRA — load failed"
print("✅ Both adapters loaded with trained weights")

Adapters available: ['v4', 'v5']
  Adapter 'v4': 260/260 lora_B modules have non-zero weights
  Adapter 'v5': 260/260 lora_B modules have non-zero weights
✅ Both adapters loaded with trained weights


In [ ]:
def collect_digit_token_ids(tokenizer, digit):
    candidates = set()
    for s in [str(digit), f" {digit}", f"\n{digit}", f"\n\n{digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            candidates.add(ids[0])
    for s in [str(digit), f" {digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) >= 1:
            candidates.add(ids[-1])
    return sorted(candidates)


CHOICE_TOKEN_VARIANTS = {}
for d in range(MAX_CHOICES):
    CHOICE_TOKEN_VARIANTS[d] = collect_digit_token_ids(student_processor.tokenizer, d)

MAX_VARIANTS = max(len(v) for v in CHOICE_TOKEN_VARIANTS.values())
CHOICE_TOKEN_MATRIX = torch.full((MAX_CHOICES, MAX_VARIANTS), -1, dtype=torch.long)
for d, ids in CHOICE_TOKEN_VARIANTS.items():
    for j, tid in enumerate(ids):
        CHOICE_TOKEN_MATRIX[d, j] = tid

CHOICE_TOKEN_MATRIX_DEVICE = None
print("CHOICE_TOKEN_MATRIX:")
print(CHOICE_TOKEN_MATRIX)


def get_choice_logits_with_active_adapter(batch):
    """Returns (B, MAX_CHOICES) choice logits using whichever adapter is currently set."""
    global CHOICE_TOKEN_MATRIX_DEVICE
    inputs = {k: (v.to(student_model.device) if torch.is_tensor(v) else v)
              for k, v in batch["inputs"].items()}
    outputs = student_model(**inputs)
    last_logits = outputs.logits[:, -1, :].float()

    if CHOICE_TOKEN_MATRIX_DEVICE is None or CHOICE_TOKEN_MATRIX_DEVICE.device != last_logits.device:
        CHOICE_TOKEN_MATRIX_DEVICE = CHOICE_TOKEN_MATRIX.to(last_logits.device)

    B = last_logits.size(0)
    C, V = CHOICE_TOKEN_MATRIX_DEVICE.shape
    flat_idx = CHOICE_TOKEN_MATRIX_DEVICE.clamp(min=0).reshape(-1)
    gathered = last_logits[:, flat_idx].reshape(B, C, V)
    valid_mask = (CHOICE_TOKEN_MATRIX_DEVICE >= 0).unsqueeze(0).expand(B, -1, -1)
    gathered = gathered.masked_fill(~valid_mask, float("-inf"))
    return torch.logsumexp(gathered, dim=-1)

CHOICE_TOKEN_MATRIX:
tensor([[32],
        [33],
        [34],
        [35],
        [36]])


In [ ]:
import torch.nn.functional as F

# ---------- v4 prompt: NO caption ----------
def build_messages_v4(row):
    hint    = safe_text(row.get("hint", ""))
    lecture = safe_text(row.get("lecture", ""))

    context_parts = []
    if lecture: context_parts.append("Lecture:\n" + lecture)
    if hint:    context_parts.append("Hint:\n" + hint)
    context_text = "\n\n".join(context_parts)

    meta_parts = []
    for col in ["grade", "subject", "topic", "category", "skill"]:
        val = safe_text(row.get(col, ""))
        if val:
            meta_parts.append(f"{col}: {val}")
    meta_text = "\n".join(meta_parts)

    choices = row["choices"]
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    text_block = f"""You are solving a science multiple-choice question.

Metadata:
{meta_text}

{context_text}

Question:
{row['question']}

Choices:
{choices_text}

Answer:""".strip()

    return [{
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": text_block}],
    }]


# ---------- v5 prompt: WITH caption (matches what v5 was trained on) ----------
MAX_LECTURE_CHARS = 800
MAX_CAPTION_CHARS = 600


def truncate_at_sentence(text, max_chars):
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    last_period = max(cut.rfind(". "), cut.rfind(".\n"), cut.rfind("! "), cut.rfind("? "))
    if last_period > max_chars * 0.5:
        return cut[:last_period + 1]
    return cut + "..."


def build_messages_v5(row):
    hint    = safe_text(row.get("hint", ""))
    lecture = safe_text(row.get("lecture", ""))
    caption = safe_text(row.get("caption", ""))

    lecture = truncate_at_sentence(lecture, MAX_LECTURE_CHARS) if lecture else ""
    caption = truncate_at_sentence(caption, MAX_CAPTION_CHARS) if caption else ""

    meta_parts = []
    for col in ["grade", "subject", "topic", "category", "skill"]:
        val = safe_text(row.get(col, ""))
        if val:
            meta_parts.append(f"{col}: {val}")
    meta_text = "\n".join(meta_parts)

    context_parts = []
    if lecture: context_parts.append("Lecture:\n" + lecture)
    if hint:    context_parts.append("Hint:\n" + hint)
    context_text = "\n\n".join(context_parts)

    choices = row["choices"]
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    sections = ["You are solving a science multiple-choice question."]
    if meta_text:    sections.append(f"Metadata:\n{meta_text}")
    if caption:      sections.append(f"Image description:\n{caption}")
    if context_text: sections.append(context_text)
    sections.append(f"Question:\n{row['question']}")
    sections.append(f"Choices:\n{choices_text}")
    sections.append("Answer:")
    text_block = "\n\n".join(sections)

    return [{
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": text_block}],
    }]


print("✅ Prompt builders ready (v4 = no caption, v5 = with caption)")

✅ Prompt builders ready (v4 = no caption, v5 = with caption)


In [ ]:
class TestEnsembleDataset(torch.utils.data.Dataset):
    """
    Each item returns the row + image. We build prompts for both adapters
    inside the inference loop because they need different prompts.
    """
    def __init__(self, df, data_dir):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        return {
            "id": row["id"],
            "row": row,
            "image": image,
            "num_choices": int(row["num_choices"]),
        }


def ensemble_collate(batch):
    return {
        "ids":         [item["id"] for item in batch],
        "rows":        [item["row"] for item in batch],
        "images":      [item["image"] for item in batch],
        "num_choices": torch.tensor([item["num_choices"] for item in batch], dtype=torch.long),
    }


@torch.no_grad()
def run_adapter_on_batch(images, prompt_messages_list, adapter_name):
    """Switch adapter, build inputs, return choice logits (B, MAX_CHOICES)."""
    student_model.set_adapter(adapter_name)
    texts = [
        student_processor.apply_chat_template(m, add_generation_prompt=True)
        for m in prompt_messages_list
    ]
    inputs = student_processor(
        text=texts, images=images, return_tensors="pt", padding=True,
    )
    return get_choice_logits_with_active_adapter({"inputs": inputs})


@torch.no_grad()
def predict_ensemble(df, weights={"v4": 0.5, "v5": 0.5}):
    """
    For each row:
      - run v4 (no-caption prompt) -> probs_v4 (after softmax)
      - run v5 (caption prompt)    -> probs_v5
      - average the probs (weighted), argmax over masked
    """
    student_model.eval()
    dataset = TestEnsembleDataset(df, DATA_DIR)
    loader  = torch.utils.data.DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=ensemble_collate, num_workers=0,
    )

    predictions = []
    for batch in tqdm(loader, desc="Ensemble inference"):
        images = batch["images"]
        rows   = batch["rows"]
        ids    = batch["ids"]
        num_choices = batch["num_choices"]

        # Run v4
        v4_messages = [build_messages_v4(r) for r in rows]
        v4_logits = run_adapter_on_batch(images, v4_messages, "v4")
        v4_probs = F.softmax(v4_logits, dim=-1)

        # Run v5
        v5_messages = [build_messages_v5(r) for r in rows]
        v5_logits = run_adapter_on_batch(images, v5_messages, "v5")
        v5_probs = F.softmax(v5_logits, dim=-1)

        # Weighted average of probs
        avg_probs = weights["v4"] * v4_probs + weights["v5"] * v5_probs

        # Mask out non-existent choices and argmax
        B, C = avg_probs.shape
        nc_dev = num_choices.to(avg_probs.device)
        arange = torch.arange(C, device=avg_probs.device).unsqueeze(0).expand(B, -1)
        mask = arange < nc_dev.unsqueeze(1)
        masked = avg_probs.masked_fill(~mask, -1.0)
        preds = masked.argmax(dim=-1).cpu().tolist()

        for sid, p in zip(ids, preds):
            predictions.append({"id": sid, "answer": int(p)})

    return pd.DataFrame(predictions)


print("✅ Ensemble prediction function ready")

✅ Ensemble prediction function ready


In [ ]:
print("Running ensemble on val to verify it beats individual models...")

val_ensemble_df = predict_ensemble(val_kd_df, weights={"v4": 0.5, "v5": 0.5})

val_compare = val_kd_df[["id", "answer"]].rename(columns={"answer": "gt"}).merge(
    val_ensemble_df.rename(columns={"answer": "ensemble_pred"}), on="id"
)

ensemble_acc = (val_compare["gt"] == val_compare["ensemble_pred"]).mean()
print(f"\nEnsemble val accuracy: {ensemble_acc*100:.2f}%")
print(f"Recall: v4 solo val ≈ 80.7%, v5 solo val ≈ similar")

Running ensemble on val to verify it beats individual models...


Ensemble inference:   0%|          | 0/524 [00:00<?, ?it/s]


Ensemble val accuracy: 82.44%
Recall: v4 solo val ≈ 80.7%, v5 solo val ≈ similar


In [ ]:
BEST_WEIGHTS = {"v4": 0.5, "v5": 0.5}   # update if E7 found better

print(f"Running ensemble on TEST with weights {BEST_WEIGHTS}...")
submission_df = predict_ensemble(test_df, weights=BEST_WEIGHTS)

print("✅ Ensemble test prediction complete | shape:", submission_df.shape)
display(submission_df.head())

submission_path = DATA_DIR / "submission_ensemble_v4v5.csv"
submission_df.to_csv(submission_path, index=False)
print("✅ Saved:", submission_path)

print("\nAnswer distribution:")
print(submission_df["answer"].value_counts().sort_index())

Running ensemble on TEST with weights {'v4': 0.5, 'v5': 0.5}...


Ensemble inference:   0%|          | 0/504 [00:00<?, ?it/s]

✅ Ensemble test prediction complete | shape: (1008, 2)


,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,0
3,test_02425,3
4,test_00930,0


✅ Saved: /content/drive/MyDrive/DL_Final_DATA/submission_ensemble_v4v5.csv

Answer distribution:
answer
0    350
1    378
2    203
3     72
4      5
Name: count, dtype: int64
